# 📰 Günlük Bülten — Colab Çalıştırıcı

Bu notebook günlük bülteni iki fazda üretir:

1. **FAZ A (`run_text.py`)** — RSS → Türkçe senaryo → 4 dile çeviri → her habere 3 görsel → klasörler.
2. **Manuel adım** — `senaryo_tr.md` dosyalarını kendi sesinle okuyup `ses/tr.wav` olarak eklersin; bir kez de `refs/ref_voice.wav` referans kaydını koyarsın.
3. **FAZ B (`run_voice.py`)** — 4 dilin sesi senin klon sesinle üretilir (Chatterbox).

Hücreleri **yukarıdan aşağıya sırayla** çalıştır.

## 1) GPU kontrolü

⚠️ **Runtime türü GPU olmalı — TPU DEĞİL.** Menü: *Runtime → Change runtime type → Hardware accelerator: **T4 GPU*** (veya L4 / A100).

Aşağıdaki hücre bir NVIDIA GPU tablosu göstermiyorsa runtime'ı değiştir ve baştan başla.

In [ ]:
!nvidia-smi
!free -h

## 2) (Opsiyonel ama önerilir) Google Drive bağla
Çıktılar, `refs/ref_voice.wav` ve `state/seen.json` oturum kapanınca silinmesin diye proje Drive'da tutulur.

In [ ]:
USE_DRIVE = True  # Drive kullanmak istemiyorsan False yap

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/gunluk-bulten'
else:
    PROJECT_DIR = '/content/gunluk-bulten'
print('Proje klasörü:', PROJECT_DIR)

## 3) Projeyi getir + temel bağımlılıkları kur
Repo özelse `REPO_URL`'e erişim token'lı adres yaz ya da dosyaları zip olarak yükleyip `PROJECT_DIR`'e aç.

In [ ]:
REPO_URL = 'https://github.com/Oguzhankuri/RSSBot.git'

if not os.path.exists(os.path.join(PROJECT_DIR, 'run_text.py')):
    !git clone {REPO_URL} "{PROJECT_DIR}"
else:
    !git -C "{PROJECT_DIR}" pull --ff-only

%cd {PROJECT_DIR}
!pip install -q -r requirements.txt

## 4) `.env` — DeepSeek anahtarı
Anahtarı [platform.deepseek.com](https://platform.deepseek.com) → *API Keys* bölümünden al. Girdiğin değer ekranda görünmez ve sadece `.env` dosyasına yazılır.

In [ ]:
from getpass import getpass

env_path = os.path.join(PROJECT_DIR, '.env')
if os.path.exists(env_path) and 'DEEPSEEK_API_KEY=sk' in open(env_path).read():
    print('.env zaten var, atlanıyor. (Değiştirmek için dosyayı sil ve hücreyi tekrar çalıştır.)')
else:
    deepseek = getpass('DEEPSEEK_API_KEY: ').strip()
    fal = getpass('FAL_KEY (images.provider=fal değilse boş bırak): ').strip()
    lines = [f'DEEPSEEK_API_KEY={deepseek}', f'FAL_KEY={fal}', '']
    with open(env_path, 'w') as fh:
        fh.write(chr(10).join(lines))
    print('.env yazıldı.')

## 5) `config.yaml` kontrolü

Sadece iki yeri doldurman yeterli:
- **`feeds`** → kendi RSS adres(ler)in
- **`translate.languages`** ve **`voice.languages`** → aynı 4 ISO dil kodu (örn. `["en", "de", "ar", "ru"]`)

Drive kullanıyorsan dosyayı soldaki 📁 panelinden açıp düzenleyebilirsin. Aşağıdaki hücre mevcut ayarları gösterir ve doğrular.

> 💡 **Görsel sağlayıcı:** Ücretsiz Colab (T4, ~12 GB RAM) FLUX'u yerelde yüklerken RAM yetmeyip çökebilir. Çökerse `images.provider: "fal"` yap ve 4. hücrede `FAL_KEY` gir.

In [ ]:
!cat config.yaml
!python -c "from bulten.config import load_config; c = load_config(require_text=True); print('✅ config geçerli — görsel sağlayıcı:', c['images']['provider'])"

## 6) FAZ A — Metin + görsel + çeviri
Önce FLUX bağımlılıkları kurulur (sadece `flux_local` ise gerekli), sonra bülten üretilir. 8 haber için T4'te ~10-20 dk sürebilir.

In [ ]:
import yaml
provider = yaml.safe_load(open('config.yaml', encoding='utf-8'))['images']['provider']
if provider == 'flux_local':
    !pip install -q -r requirements-gpu.txt
!python run_text.py

## 7) ✋ Manuel adım — Türkçe seslendirme

1. `output/<bugünün tarihi>/` altındaki her `NN-.../senaryo_tr.md` dosyasını **kendi sesinle** oku.
2. Her kaydı ilgili haberin `ses/tr.wav` dosyası olarak yükle.
3. **Bir kez:** 10–20 saniyelik, gürültüsüz, tek kişilik konuşma kaydını `refs/ref_voice.wav` olarak yükle (klon ses bundan üretilir).

Hazır olunca aşağıya devam et.

In [ ]:
import glob
print('Referans ses:', '✅ var' if os.path.exists('refs/ref_voice.wav') else '❌ YOK — refs/ref_voice.wav yükle')
for d in sorted(glob.glob('output/*/[0-9][0-9]-*')):
    print(('✅ ' if os.path.exists(f'{d}/ses/tr.wav') else '⏳ ') + d)

## 8) FAZ B — 4 dil seslendirme (Chatterbox)
Chatterbox kendi `torch`/`diffusers` sürümlerini kurar; bu yüzden FAZ A'dan **sonra** kurulur. İlk kurulum birkaç dakika sürer.

Farklı bir günü seslendirmek için: `!python run_voice.py --date 2026-10-08`

In [ ]:
!pip install -q -r requirements-voice.txt
!python run_voice.py

## 9) Çıktıyı indir
Drive kullanıyorsan dosyalar zaten `MyDrive/gunluk-bulten/output/` içinde. Değilse aşağıdaki hücre günün klasörünü zip'leyip indirir.

In [ ]:
import shutil, datetime
from zoneinfo import ZoneInfo
today = datetime.datetime.now(ZoneInfo('Europe/Istanbul')).strftime('%Y-%m-%d')
zip_path = shutil.make_archive(f'/content/bulten-{today}', 'zip', f'output/{today}')
print('Zip:', zip_path)
if not USE_DRIVE:
    from google.colab import files
    files.download(zip_path)